## Stationarity Test

For linear regression approaches, it is good to see if our data has stationarity. If not, we must visualize and figure out the trend/seasonal trend occuring and feature engineer our data to achieve stationarity.

On the other hand, this step does not matter too much when training neural networks, as they will take advantage of the patterns. It is important to keep time embedding features, spatial features, and station id for as much information as possible.

In [ ]:
from statsmodels.tsa.stattools import adfuller
from pathlib import Path

In [ ]:
def adf_test_multiple_periods(df, col_name, periods=None):

    if periods is None:
        start_date = df.index.min()
        end_date = df.index.max()
        
        # default periods
        periods = [
            ('Last Month', end_date - pd.Timedelta(days=30), end_date),
            ('Last Year', end_date - pd.Timedelta(days=365), end_date),
            ('Total Period', start_date, end_date)
        ]
    
    for period_name, start, end in periods:
        period_data = df.loc[start:end, col_name].dropna()
        
        if len(period_data) < 2:
            print(f"\nNot enough data for {period_name} period")
            continue
            
        result = adfuller(period_data)
        
        print(f"\nADF Test Results for {col_name} - {period_name}")
        print(f'ADF Statistic: {result[0]:.4f}')
        print(f'p-value: {result[1]:.4f}')
        print('Critical Values:')
        for key, value in result[4].items():
            print(f'  {key}: {value:.4f}')
        
        # See if data is stationary or not
        if result[1] <= 0.05:
            print(f"Conclusion: Stationary (reject H0)")
        else:
            print(f"Conclusion: Non-stationary (fail to reject H0)")

In [ ]:
base_dir = Path("../data-cleanup/cleaned_data")
file_path = base_dir / "Station1_cleaned_data.csv"

df = pd.read_csv(file_path, index_col=0, parse_dates=True)
df.drop('Flag', axis=1, inplace=True)

In [ ]:
custom_periods = [
    ('Spring 2019', '2019-03-01', '2019-05-31'),
    ('Summer 2019', '2019-06-01', '2019-08-31'),
    ('Fall 2019', '2019-09-01', '2019-11-30'),
    ('Winter 2019', '2019-12-01', '2020-02-28'),
    ('2019', '2019-01-01', '2019-12-31'),
    ('Total Period', '2015-01-01', '2020-12-31')
]
for col in df.columns:
    
    adf_test_multiple_periods(df, str(col), periods=custom_periods)